In [1]:
%pip install -U azure-ai-projects azure-identity python-dotenv numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from app.azure_data import load_data

employees = load_data()

print(f"Loaded {len(employees)} employees")
employees

Loaded 3 employees


[{'id': 0,
  'name': 'Adam Olderr',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'expert',
   'dotnet': 'advanced',
   'python': 'advanced',
   'sql': 'advanced',
   'pulumi': 'novice'}},
 {'id': 1,
  'name': 'Willaim Javier',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'advanced',
   'dotnet': 'intermediate',
   'python': 'advanced',
   'sql': 'intermediate',
   'pulumi': 'intermediate'}},
 {'id': 2,
  'name': 'Priyanka Aggarwal',
  'position': 'Senior Software Engineer',
  'skill_map': {'angular': 'intermediate',
   'dotnet': 'advanced',
   'python': 'expert',
   'sql': 'advanced',
   'pulumi': 'expert'}}]

In [3]:
def employee_to_text(employee):
    skills = ", ".join(
        f"{skill}: {level}"
        for skill, level in employee["skill_map"].items()
    )

    return (
        f"Employee: {employee['name']}. "
        f"Position: {employee['position']}. "
        f"Skills and proficiency: {skills}."
    )

employee_texts = [employee_to_text(e) for e in employees]

print(employee_texts[0])

Employee: Adam Olderr. Position: Software Engineer. Skills and proficiency: angular: expert, dotnet: advanced, python: advanced, sql: advanced, pulumi: novice.


In [4]:
import os

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

# Use your Foundry resource's OpenAI endpoint,
# not the project endpoint ending in /api/projects/...
embedding_client = OpenAI(
    base_url="https://foundry-aoi-1993.openai.azure.com/openai/v1",
    api_key=token_provider,
)

embedding_model = os.environ["EMBEDDING_MODEL_NAME"]

In [5]:
import os

print("Embedding deployment:", repr(embedding_model))
print("Client base URL:", embedding_client.base_url)

Embedding deployment: 'text-embedding-3-small'
Client base URL: https://foundry-aoi-1993.openai.azure.com/openai/v1/


In [6]:
import numpy as np

response = embedding_client.embeddings.create(
    model=embedding_model,
    input=employee_texts,
)

# Keep vectors in the same order as employee_texts.
embeddings = np.array(
    [item.embedding for item in sorted(
        response.data,
        key=lambda item: item.index,
    )],
    dtype=np.float32,
)

print("Number of employee vectors:", embeddings.shape[0])
print("Dimensions per vector:", embeddings.shape[1])
print("First 10 values of the first vector:", embeddings[0][:10])

Number of employee vectors: 3
Dimensions per vector: 1536
First 10 values of the first vector: [-0.00020194 -0.01194     0.04766846 -0.01229858  0.01626587 -0.00260353
 -0.01074982  0.04370117 -0.01167297 -0.00785065]


In [12]:
task = "Build a UI component in Angular."

query_response = embedding_client.embeddings.create(
    model=embedding_model,
    input=[task],
)

query_vector = np.array(
    query_response.data[0].embedding,
    dtype=np.float32,
)

def cosine_similarity(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator == 0:
        return 0.0

    return float(np.dot(a, b) / denominator)


scores = [
    cosine_similarity(query_vector, vector)
    for vector in embeddings
]

ranked_indices = np.argsort(scores)[::-1]

for index in ranked_indices[:3]:
    print(f"{scores[index]:.4f} | {employee_texts[index]}")

0.2480 | Employee: Willaim Javier. Position: Software Engineer. Skills and proficiency: angular: advanced, dotnet: intermediate, python: advanced, sql: intermediate, pulumi: intermediate.
0.2466 | Employee: Priyanka Aggarwal. Position: Senior Software Engineer. Skills and proficiency: angular: intermediate, dotnet: advanced, python: expert, sql: advanced, pulumi: expert.
0.2311 | Employee: Adam Olderr. Position: Software Engineer. Skills and proficiency: angular: expert, dotnet: advanced, python: advanced, sql: advanced, pulumi: novice.


In [13]:
embedding_client.close()
credential.close()